# State-Dependent U.S. Equity Sector Rotation
## A Systematic Framework for Trend-Based Active Sector Allocation

# Block 4 — State Machine & State-Dependent Active-Weight Engine

Block 4 translates the validated Block 3 indicator series into persistent weekly sector states and portfolio tilts.

Two design rules are enforced:

1. **Pine state-machine parity.** The supplied TradingView Pine Script remains the source of truth for zone memory, entry priority, exit rules, and strategic position events.
2. **Portfolio-state separation.** Tactical pullback/profit-taking regimes are kept separate from the underlying strategic trend state.

The initial active-weight staircase is:

- bearish → maximum underweight;
- early reversal → rebuild toward neutral by 0.10 per week;
- confirmed reversal → neutral;
- pullback accumulation → increase toward maximum overweight by 0.10 per week;
- profit taking → reduce an existing overweight toward neutral by 0.10 per week.

The active multiplier is bounded at **−0.50 to +0.50**.

Portfolio weights are formed as:

\[
w^{raw}_{i,t}=w^{Strategic}_{i,t}(1+A_{i,t})
\]

and normalized cross-sectionally to sum to one.


In [1]:
# ============================================================
# BLOCK 4.1 — ENVIRONMENT, DRIVE & PRIOR MANIFESTS
# ============================================================

from pathlib import Path
from google.colab import drive
import json
from datetime import datetime, timezone

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation")

DIRS = {
    "root": PROJECT_ROOT,
    "data_processed": PROJECT_ROOT / "data" / "processed",
    "outputs": PROJECT_ROOT / "outputs",
    "tables": PROJECT_ROOT / "outputs" / "tables",
    "manifests": PROJECT_ROOT / "manifests",
    "logs": PROJECT_ROOT / "logs",
}

for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

BLOCK1_MANIFEST = DIRS["manifests"] / "block_1_research_configuration.json"
BLOCK2_MANIFEST = DIRS["manifests"] / "block_2_universe_data_weights.json"
BLOCK3_MANIFEST = DIRS["manifests"] / "block_3_signal_engine.json"

for p in [BLOCK1_MANIFEST, BLOCK2_MANIFEST, BLOCK3_MANIFEST]:
    if not p.exists():
        raise FileNotFoundError(f"Required prior manifest not found: {p}")

with open(BLOCK1_MANIFEST, "r", encoding="utf-8") as f:
    block1 = json.load(f)

with open(BLOCK2_MANIFEST, "r", encoding="utf-8") as f:
    block2 = json.load(f)

with open(BLOCK3_MANIFEST, "r", encoding="utf-8") as f:
    block3 = json.load(f)

CONFIG = block1["research_config"]
SECTOR_RECORDS = block1["sector_universe"]
SECTOR_TICKERS = [x["ticker"] for x in SECTOR_RECORDS]

assert block2["strategic_weight_method"] == "INVERSE_VOLATILITY_52W"
assert block3["methodology"]["state_machine_included"] is False
assert block3["indicator_source"]["source_of_truth"] is True
assert CONFIG["signal_frequency"] == "W-FRI"
assert CONFIG["rebalance_execution_rule"] == "NEXT_US_TRADING_SESSION_AFTER_SIGNAL"

print("Loaded and validated Blocks 1–3.")
print("Strategic allocation:", block2["strategic_weight_method"])
print("Indicator source:", block3["indicator_source"]["name"])
print(
    "Canonical signal window:",
    block3["canonical_signal_start"],
    "to",
    block3["canonical_signal_end"],
)


Mounted at /content/drive
Loaded and validated Blocks 1–3.
Strategic allocation: INVERSE_VOLATILITY_52W
Indicator source: Trend Following SuperSmoother - Accumulation Zones [JW]
Canonical signal window: 2019-06-21 to 2026-08-21


In [2]:
# ============================================================
# BLOCK 4.2 — IMPORTS & LOAD BLOCK 3 / STRATEGIC WEIGHT DATA
# ============================================================

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 190)

SIGNAL_LONG_PATH = Path(block3["saved_files"]["signal_engine_long"])
STRATEGIC_WEIGHT_PATH = Path(block2["saved_files"]["strategic_weights"])

for p in [SIGNAL_LONG_PATH, STRATEGIC_WEIGHT_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"Required dataset not found: {p}")

signal_long = pd.read_parquet(SIGNAL_LONG_PATH)
strategic_weights = pd.read_parquet(STRATEGIC_WEIGHT_PATH)

signal_long["date"] = pd.to_datetime(signal_long["date"])
signal_long["execution_date"] = pd.to_datetime(signal_long["execution_date"])

strategic_weights.index = pd.to_datetime(strategic_weights.index)
strategic_weights.index.name = "date"

required_signal_columns = {
    "date",
    "ticker",
    "oscillator",
    "signal_line",
    "upper_bb",
    "lower_bb",
    "osc_rising",
    "osc_falling",
    "osc_turns_red",
    "cross_below_zero",
    "signal_cross_above_zero",
    "osc_cross_above_signal",
    "osc_cross_above_lower_bb",
    "osc_cross_below_upper_bb",
    "indicator_ready",
    "execution_date",
}

missing = required_signal_columns.difference(signal_long.columns)
if missing:
    raise ValueError(f"Block 3 signal dataset missing columns: {sorted(missing)}")

assert set(SECTOR_TICKERS).issubset(strategic_weights.columns)

print(f"Signal rows loaded: {len(signal_long):,}")
print(f"Strategic-weight weeks loaded: {len(strategic_weights):,}")


Signal rows loaded: 4,708
Strategic-weight weeks loaded: 376


In [3]:
# ============================================================
# BLOCK 4.3 — ACTIVE-WEIGHT PARAMETERS
# ============================================================

ACTIVE_MIN = float(CONFIG["active_weight_min"])
ACTIVE_MAX = float(CONFIG["active_weight_max"])
ACTIVE_STEP = float(CONFIG["active_step"])

assert ACTIVE_MIN == -0.50
assert ACTIVE_MAX == 0.50
assert ACTIVE_STEP == 0.10
assert 1.0 + ACTIVE_MIN > 0.0

active_parameters = pd.Series(
    {
        "Minimum active multiplier": ACTIVE_MIN,
        "Maximum active multiplier": ACTIVE_MAX,
        "Weekly staircase step": ACTIVE_STEP,
        "Minimum raw weight multiplier": 1.0 + ACTIVE_MIN,
        "Neutral raw weight multiplier": 1.0,
        "Maximum raw weight multiplier": 1.0 + ACTIVE_MAX,
    },
    name="Block 4 parameters",
).to_frame()

display(active_parameters)


,Block 4 parameters
Minimum active multiplier,-0.5
Maximum active multiplier,0.5
Weekly staircase step,0.1
Minimum raw weight multiplier,0.5
Neutral raw weight multiplier,1.0
Maximum raw weight multiplier,1.5


## State representation

The Pine logic retains its exact persistent `zoneMode`:

- `0 = neutral`
- `1 = early reversal`
- `2 = pullback accumulation`
- `3 = profit taking`

with priority **early reversal > pullback > profit taking**.

For research and portfolio construction, Block 4 exposes three separate fields:

- **`strategic_state`** — `BEARISH`, `EARLY_REVERSAL`, `CONFIRMED_TREND`, or `NORMAL_POSITIVE_TREND`;
- **`tactical_state`** — the **portfolio-effective** overlay: `NONE`, `PULLBACK_ACCUMULATION`, or `PROFIT_TAKING`; it is actionable only while the strategic Pine position is held;
- **`pine_zone_state`** — exact persistent Pine zone state for parity/audit.

This prevents a tactical overlay from silently overwriting the strategic trend state.


The raw Pine zone is still retained even when the strategic position is OFF. This matters because Pine zone logic and strategic holding logic are separate. For portfolio construction, however, a pullback/profit-taking zone cannot create an allocation action while the strategic position is bearish/off.

In [4]:
# ============================================================
# BLOCK 4.4 — PINE-PARITY SINGLE-SECTOR STATE MACHINE
# ============================================================

ZONE_NAMES = {
    0: "NEUTRAL",
    1: "EARLY_REVERSAL",
    2: "PULLBACK_ACCUMULATION",
    3: "PROFIT_TAKING",
}


def run_state_machine_for_sector(df: pd.DataFrame) -> pd.DataFrame:
    """
    Stateful translation of the supplied Pine zone and position logic.

    Input: one ticker's Block 3 rows in chronological order.
    Output: persistent states and one-bar transition events.
    """

    df = df.sort_values("date").reset_index(drop=True)

    zone_mode = 0
    early_reversal_exit_mode = 0
    early_reversal_saw_below_bb = False
    pullback_saw_above_bb = False
    position_held = False

    rows = []

    for _, r in df.iterrows():
        oscillator = float(r["oscillator"])
        signal_line = float(r["signal_line"])
        upper_bb = r["upper_bb"]
        lower_bb = r["lower_bb"]

        osc_rising = bool(r["osc_rising"])
        osc_falling = bool(r["osc_falling"])
        osc_turns_red = bool(r["osc_turns_red"])

        cross_below_zero = bool(r["cross_below_zero"])
        signal_cross_above_zero = bool(r["signal_cross_above_zero"])
        osc_cross_above_signal = bool(r["osc_cross_above_signal"])
        osc_cross_above_lower_bb = bool(r["osc_cross_above_lower_bb"])
        osc_cross_below_upper_bb = bool(r["osc_cross_below_upper_bb"])

        start_early_reversal = False
        start_pullback = False
        end_early_reversal = False
        end_pullback = False
        start_profit_taking = False
        end_profit_taking = False
        position_entry_event = False
        position_exit_event = False

        # ----------------------------------------------------
        # PRE-TRANSITION HELPERS
        # ----------------------------------------------------
        early_reversal_active_before = zone_mode == 1
        pullback_active_before = zone_mode == 2
        accumulation_active_before = (
            early_reversal_active_before or pullback_active_before
        )
        can_search_for_entry = not accumulation_active_before

        early_reversal_entry_intersection = (
            early_reversal_saw_below_bb
            and oscillator < 0
            and osc_cross_above_lower_bb
        )

        pullback_entry_intersection = (
            pullback_saw_above_bb
            and oscillator > 0
            and osc_cross_below_upper_bb
        )

        # ----------------------------------------------------
        # ONE-SHOT SETUP MEMORY
        # ----------------------------------------------------
        if can_search_for_entry:
            if (
                pd.notna(lower_bb)
                and oscillator < 0
                and oscillator < lower_bb
            ):
                early_reversal_saw_below_bb = True

            if oscillator >= 0:
                early_reversal_saw_below_bb = False

            if (
                pd.notna(upper_bb)
                and oscillator > 0
                and oscillator > upper_bb
            ):
                pullback_saw_above_bb = True

            if oscillator <= 0:
                pullback_saw_above_bb = False
        else:
            early_reversal_saw_below_bb = False
            pullback_saw_above_bb = False

        # ----------------------------------------------------
        # ENTRY CONDITIONS
        # ----------------------------------------------------
        early_reversal_entry = (
            can_search_for_entry
            and early_reversal_saw_below_bb
            and pd.notna(lower_bb)
            and oscillator < 0
            and oscillator > lower_bb
            and osc_rising
        )

        pullback_entry = (
            can_search_for_entry
            and pullback_saw_above_bb
            and pd.notna(upper_bb)
            and oscillator > 0
            and oscillator < upper_bb
            and osc_falling
        )

        profit_taking_entry = (
            can_search_for_entry
            and not early_reversal_entry
            and not pullback_entry
            and oscillator > 0
            and osc_turns_red
        )

        # ----------------------------------------------------
        # ZONE STATE MACHINE
        # ----------------------------------------------------
        if zone_mode == 1:
            if (
                early_reversal_exit_mode == 1
                and signal_cross_above_zero
            ):
                zone_mode = 0
                early_reversal_exit_mode = 0
                end_early_reversal = True

            elif (
                early_reversal_exit_mode == 2
                and osc_cross_above_signal
            ):
                zone_mode = 0
                early_reversal_exit_mode = 0
                end_early_reversal = True

        elif zone_mode == 2:
            if osc_rising:
                zone_mode = 0
                end_pullback = True

            elif oscillator < 0:
                zone_mode = 0
                end_pullback = True

        else:
            if early_reversal_entry:
                if zone_mode == 3:
                    end_profit_taking = True

                zone_mode = 1
                start_early_reversal = True

                if signal_line < 0:
                    early_reversal_exit_mode = 1
                else:
                    early_reversal_exit_mode = 2

                early_reversal_saw_below_bb = False
                pullback_saw_above_bb = False

            elif pullback_entry:
                if zone_mode == 3:
                    end_profit_taking = True

                zone_mode = 2
                start_pullback = True
                early_reversal_exit_mode = 0
                pullback_saw_above_bb = False
                early_reversal_saw_below_bb = False

            elif zone_mode == 3:
                if cross_below_zero:
                    zone_mode = 0
                    end_profit_taking = True

            elif profit_taking_entry:
                zone_mode = 3
                start_profit_taking = True

        # ----------------------------------------------------
        # STRATEGIC POSITION STATE MACHINE
        # ----------------------------------------------------
        if position_held and cross_below_zero:
            position_held = False
            position_exit_event = True

        elif (not position_held) and start_early_reversal:
            position_held = True
            position_entry_event = True

        # ----------------------------------------------------
        # RESEARCH-FACING STATE FIELDS
        # ----------------------------------------------------
        early_reversal_active = zone_mode == 1
        pullback_active = zone_mode == 2
        profit_taking_active = zone_mode == 3

        if not position_held:
            strategic_state = "BEARISH"
        elif early_reversal_active:
            strategic_state = "EARLY_REVERSAL"
        elif end_early_reversal:
            strategic_state = "CONFIRMED_TREND"
        else:
            strategic_state = "NORMAL_POSITIVE_TREND"

        # Portfolio-effective tactical overlay.
        #
        # Pine can enter pullback/profit-taking zones even while its separate
        # strategic `positionHeld` state is OFF. We preserve that raw Pine
        # information in `pine_zone_state`, but tactical allocation changes
        # are actionable only while the strategic position is held.
        if not position_held:
            tactical_state = "NONE"
        elif pullback_active:
            tactical_state = "PULLBACK_ACCUMULATION"
        elif profit_taking_active:
            tactical_state = "PROFIT_TAKING"
        else:
            tactical_state = "NONE"

        rows.append(
            {
                "date": r["date"],
                "ticker": r["ticker"],
                "pine_zone_mode": zone_mode,
                "pine_zone_state": ZONE_NAMES[zone_mode],
                "early_reversal_exit_mode": early_reversal_exit_mode,
                "early_reversal_saw_below_bb": early_reversal_saw_below_bb,
                "pullback_saw_above_bb": pullback_saw_above_bb,
                "position_held": position_held,
                "strategic_state": strategic_state,
                "tactical_state": tactical_state,
                "early_reversal_entry_intersection":
                    early_reversal_entry_intersection,
                "pullback_entry_intersection":
                    pullback_entry_intersection,
                "start_early_reversal": start_early_reversal,
                "end_early_reversal": end_early_reversal,
                "start_pullback": start_pullback,
                "end_pullback": end_pullback,
                "start_profit_taking": start_profit_taking,
                "end_profit_taking": end_profit_taking,
                "position_entry_event": position_entry_event,
                "position_exit_event": position_exit_event,
            }
        )

    return pd.DataFrame(rows)


In [5]:
# ============================================================
# BLOCK 4.5 — RUN STATE MACHINE INDEPENDENTLY FOR 11 SECTORS
# ============================================================

state_frames = []

for ticker in SECTOR_TICKERS:
    sector_input = signal_long[
        signal_long["ticker"] == ticker
    ].copy()

    if sector_input.empty:
        raise RuntimeError(f"No Block 3 signal rows found for {ticker}")

    state_frames.append(run_state_machine_for_sector(sector_input))

state_long = (
    pd.concat(state_frames, ignore_index=True)
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

expected_rows = (
    signal_long[signal_long["ticker"].isin(SECTOR_TICKERS)]
    [["date", "ticker"]]
    .drop_duplicates()
)

assert len(state_long) == len(expected_rows)
assert not state_long.duplicated(["date", "ticker"]).any()

print(f"State-machine rows: {len(state_long):,}")
display(state_long.head(15))


State-machine rows: 4,708


,date,ticker,pine_zone_mode,pine_zone_state,early_reversal_exit_mode,early_reversal_saw_below_bb,pullback_saw_above_bb,position_held,strategic_state,tactical_state,early_reversal_entry_intersection,pullback_entry_intersection,start_early_reversal,end_early_reversal,start_pullback,end_pullback,start_profit_taking,end_profit_taking,position_entry_event,position_exit_event
0,2018-06-22,XLB,0,NEUTRAL,0,False,False,False,BEARISH,NONE,False,False,False,False,False,False,False,False,False,False
1,2018-06-22,XLC,0,NEUTRAL,0,False,False,False,BEARISH,NONE,False,False,False,False,False,False,False,False,False,False
2,2018-06-22,XLE,0,NEUTRAL,0,False,False,False,BEARISH,NONE,False,False,False,False,False,False,False,False,False,False
3,2018-06-22,XLF,0,NEUTRAL,0,False,False,False,BEARISH,NONE,False,False,False,False,False,False,False,False,False,False
4,2018-06-22,XLI,0,NEUTRAL,0,False,False,False,BEARISH,NONE,False,False,False,False,False,False,False,False,False,False
5,2018-06-22,XLK,0,NEUTRAL,0,False,False,False,BEARISH,NONE,False,False,False,False,False,False,False,False,False,False
6,2018-06-22,XLP,0,NEUTRAL,0,False,False,False,BEARISH,NONE,False,False,False,False,False,False,False,False,False,False
7,2018-06-22,XLRE,0,NEUTRAL,0,False,False,False,BEARISH,NONE,False,False,False,False,False,False,False,False,False,False
8,2018-06-22,XLU,0,NEUTRAL,0,False,False,False,BEARISH,NONE,False,False,False,False,False,False,False,False,False,False
9,2018-06-22,XLV,0,NEUTRAL,0,False,False,False,BEARISH,NONE,False,False,False,False,False,False,False,False,False,False


In [6]:
# ============================================================
# BLOCK 4.6 — STATE / EVENT DIAGNOSTICS
# ============================================================

strategic_counts = (
    state_long["strategic_state"]
    .value_counts()
    .rename("week_sector_rows")
    .to_frame()
)

tactical_counts = (
    state_long["tactical_state"]
    .value_counts()
    .rename("week_sector_rows")
    .to_frame()
)

event_columns = [
    "start_early_reversal",
    "end_early_reversal",
    "start_pullback",
    "end_pullback",
    "start_profit_taking",
    "end_profit_taking",
    "position_entry_event",
    "position_exit_event",
]

event_counts = (
    state_long[event_columns]
    .sum()
    .astype(int)
    .rename("event_count")
    .to_frame()
)

print("Strategic states:")
display(strategic_counts)

print("Tactical states:")
display(tactical_counts)

print("State-machine events:")
display(event_counts)


Strategic states:


,week_sector_rows
strategic_state,
BEARISH,2628
NORMAL_POSITIVE_TREND,1698
EARLY_REVERSAL,366
CONFIRMED_TREND,16


Tactical states:


,week_sector_rows
tactical_state,
NONE,3519
PROFIT_TAKING,884
PULLBACK_ACCUMULATION,305


State-machine events:


,event_count
start_early_reversal,17
end_early_reversal,17
start_pullback,80
end_pullback,80
start_profit_taking,100
end_profit_taking,92
position_entry_event,17
position_exit_event,8


## Active-weight staircase

The indicator state and portfolio allocation are deliberately separated.

- **BEARISH:** set active multiplier to `−0.50`.
- **EARLY REVERSAL:** add `+0.10` per active reversal week, capped at neutral.
- **CONFIRMED TREND:** move immediately to neutral if confirmation arrives before rebuilding finishes.
- **PULLBACK ACCUMULATION:** add `+0.10` per week, capped at `+0.50`.
- **PROFIT TAKING:** subtract `0.10` from an existing positive overweight, floored at neutral.
- **NORMAL POSITIVE TREND:** retain the current positive/neutral tilt until another state changes it.

Profit taking cannot create an underweight, and pullback accumulation cannot override a bearish strategic state.


In [7]:
# ============================================================
# BLOCK 4.7 — STATE-DEPENDENT ACTIVE MULTIPLIER ENGINE
# ============================================================

def run_active_weight_engine_for_sector(df: pd.DataFrame) -> pd.DataFrame:
    """
    Translate states into active multiplier A_t in [-0.50, +0.50].
    """

    df = df.sort_values("date").reset_index(drop=True)

    active = ACTIVE_MIN
    rows = []

    for _, r in df.iterrows():
        prev_active = active

        strategic = r["strategic_state"]
        tactical = r["tactical_state"]

        if strategic == "BEARISH":
            active = ACTIVE_MIN
            action = "SET_MAX_UNDERWEIGHT"

        elif strategic == "CONFIRMED_TREND":
            active = 0.0
            action = "SET_NEUTRAL_ON_CONFIRMATION"

        elif strategic == "EARLY_REVERSAL":
            active = min(
                0.0,
                max(ACTIVE_MIN, active) + ACTIVE_STEP,
            )
            action = "REBUILD_TOWARD_NEUTRAL"

        elif tactical == "PULLBACK_ACCUMULATION":
            active = min(
                ACTIVE_MAX,
                max(0.0, active) + ACTIVE_STEP,
            )
            action = "ADD_OVERWEIGHT"

        elif tactical == "PROFIT_TAKING":
            active = max(
                0.0,
                active - ACTIVE_STEP,
            )
            action = "REDUCE_OVERWEIGHT"

        else:
            active = float(np.clip(active, 0.0, ACTIVE_MAX))
            action = "HOLD_POSITIVE_TILT"

        active = float(np.clip(active, ACTIVE_MIN, ACTIVE_MAX))

        rows.append(
            {
                "date": r["date"],
                "ticker": r["ticker"],
                "previous_active_multiplier": prev_active,
                "active_multiplier": active,
                "active_weight_action": action,
            }
        )

    return pd.DataFrame(rows)


active_frames = []

for ticker in SECTOR_TICKERS:
    sector_states = state_long[
        state_long["ticker"] == ticker
    ].copy()

    active_frames.append(
        run_active_weight_engine_for_sector(sector_states)
    )

active_long = (
    pd.concat(active_frames, ignore_index=True)
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

assert len(active_long) == len(state_long)
assert active_long["active_multiplier"].between(
    ACTIVE_MIN,
    ACTIVE_MAX,
).all()

display(active_long.head(15))


,date,ticker,previous_active_multiplier,active_multiplier,active_weight_action
0,2018-06-22,XLB,-0.5,-0.5,SET_MAX_UNDERWEIGHT
1,2018-06-22,XLC,-0.5,-0.5,SET_MAX_UNDERWEIGHT
2,2018-06-22,XLE,-0.5,-0.5,SET_MAX_UNDERWEIGHT
3,2018-06-22,XLF,-0.5,-0.5,SET_MAX_UNDERWEIGHT
4,2018-06-22,XLI,-0.5,-0.5,SET_MAX_UNDERWEIGHT
5,2018-06-22,XLK,-0.5,-0.5,SET_MAX_UNDERWEIGHT
6,2018-06-22,XLP,-0.5,-0.5,SET_MAX_UNDERWEIGHT
7,2018-06-22,XLRE,-0.5,-0.5,SET_MAX_UNDERWEIGHT
8,2018-06-22,XLU,-0.5,-0.5,SET_MAX_UNDERWEIGHT
9,2018-06-22,XLV,-0.5,-0.5,SET_MAX_UNDERWEIGHT


In [8]:
# ============================================================
# BLOCK 4.8 — MERGE STATES, SIGNALS & ACTIVE MULTIPLIERS
# ============================================================

state_signal_long = (
    signal_long
    .merge(
        state_long,
        on=["date", "ticker"],
        how="left",
        validate="one_to_one",
    )
    .merge(
        active_long,
        on=["date", "ticker"],
        how="left",
        validate="one_to_one",
    )
    .sort_values(["date", "ticker"])
    .reset_index(drop=True)
)

required_state_outputs = [
    "strategic_state",
    "tactical_state",
    "pine_zone_state",
    "position_held",
    "active_multiplier",
]

assert state_signal_long[required_state_outputs].notna().all().all()

print(f"Merged state/signal rows: {len(state_signal_long):,}")
display(
    state_signal_long[
        [
            "date",
            "ticker",
            "oscillator",
            "signal_line",
            "strategic_state",
            "tactical_state",
            "pine_zone_state",
            "position_held",
            "active_multiplier",
            "active_weight_action",
        ]
    ].head(20)
)


Merged state/signal rows: 4,708


,date,ticker,oscillator,signal_line,strategic_state,tactical_state,pine_zone_state,position_held,active_multiplier,active_weight_action
0,2018-06-22,XLB,0.000000,0.000000,BEARISH,NONE,NEUTRAL,False,-0.5,SET_MAX_UNDERWEIGHT
1,2018-06-22,XLC,0.000000,0.000000,BEARISH,NONE,NEUTRAL,False,-0.5,SET_MAX_UNDERWEIGHT
2,2018-06-22,XLE,0.000000,0.000000,BEARISH,NONE,NEUTRAL,False,-0.5,SET_MAX_UNDERWEIGHT
3,2018-06-22,XLF,0.000000,0.000000,BEARISH,NONE,NEUTRAL,False,-0.5,SET_MAX_UNDERWEIGHT
4,2018-06-22,XLI,0.000000,0.000000,BEARISH,NONE,NEUTRAL,False,-0.5,SET_MAX_UNDERWEIGHT
5,2018-06-22,XLK,0.000000,0.000000,BEARISH,NONE,NEUTRAL,False,-0.5,SET_MAX_UNDERWEIGHT
6,2018-06-22,XLP,0.000000,0.000000,BEARISH,NONE,NEUTRAL,False,-0.5,SET_MAX_UNDERWEIGHT
7,2018-06-22,XLRE,0.000000,0.000000,BEARISH,NONE,NEUTRAL,False,-0.5,SET_MAX_UNDERWEIGHT
8,2018-06-22,XLU,0.000000,0.000000,BEARISH,NONE,NEUTRAL,False,-0.5,SET_MAX_UNDERWEIGHT
9,2018-06-22,XLV,0.000000,0.000000,BEARISH,NONE,NEUTRAL,False,-0.5,SET_MAX_UNDERWEIGHT


In [9]:
# ============================================================
# BLOCK 4.9 — STRATEGIC + ACTIVE RAW / NORMALIZED TARGET WEIGHTS
# ============================================================

strategic_long = (
    strategic_weights[SECTOR_TICKERS]
    .rename_axis(index="date", columns="ticker")
    .stack(future_stack=True)
    .rename("strategic_weight")
    .reset_index()
)

weight_long = (
    state_signal_long
    .merge(
        strategic_long,
        on=["date", "ticker"],
        how="left",
        validate="many_to_one",
    )
)

weight_long["raw_weight_multiplier"] = (
    1.0 + weight_long["active_multiplier"]
)

weight_long["raw_target_weight"] = (
    weight_long["strategic_weight"]
    * weight_long["raw_weight_multiplier"]
)

raw_sum = (
    weight_long
    .groupby("date")["raw_target_weight"]
    .transform("sum")
)

weight_long["normalized_target_weight"] = (
    weight_long["raw_target_weight"] / raw_sum
)

canonical_start = pd.Timestamp(block3["canonical_signal_start"])
canonical_end = pd.Timestamp(block3["canonical_signal_end"])

canonical_weights = weight_long[
    (weight_long["date"] >= canonical_start)
    & (weight_long["date"] <= canonical_end)
].copy()

canonical_complete = canonical_weights[
    [
        "strategic_weight",
        "active_multiplier",
        "raw_target_weight",
        "normalized_target_weight",
        "execution_date",
    ]
].notna().all(axis=1)

if not canonical_complete.all():
    bad = canonical_weights.loc[
        ~canonical_complete,
        ["date", "ticker"]
    ]
    display(bad.head(20))
    raise ValueError("Incomplete canonical Block 4 weight rows detected.")

target_sums = (
    canonical_weights
    .groupby("date")["normalized_target_weight"]
    .sum()
)

assert np.allclose(target_sums.values, 1.0, atol=1e-10)
assert (canonical_weights["normalized_target_weight"] >= 0).all()

print(
    "Canonical complete rows:",
    f"{canonical_complete.sum():,} / {len(canonical_complete):,}"
)
print(
    "Normalized target-weight sum range:",
    f"{target_sums.min():.12f} to {target_sums.max():.12f}"
)


Canonical complete rows: 4,125 / 4,125
Normalized target-weight sum range: 1.000000000000 to 1.000000000000


In [10]:
# ============================================================
# BLOCK 4.10 — CANONICAL STATE & ACTIVE-WEIGHT DIAGNOSTICS
# ============================================================

canonical_state_counts = (
    canonical_weights["strategic_state"]
    .value_counts()
    .rename("week_sector_rows")
    .to_frame()
)

canonical_tactical_counts = (
    canonical_weights["tactical_state"]
    .value_counts()
    .rename("week_sector_rows")
    .to_frame()
)

canonical_action_counts = (
    canonical_weights["active_weight_action"]
    .value_counts()
    .rename("week_sector_rows")
    .to_frame()
)

canonical_event_counts = (
    canonical_weights[event_columns]
    .sum()
    .astype(int)
    .rename("event_count")
    .to_frame()
)

print("Canonical strategic-state distribution:")
display(canonical_state_counts)

print("Canonical tactical-state distribution:")
display(canonical_tactical_counts)

print("Canonical active-weight actions:")
display(canonical_action_counts)

print("Canonical event counts:")
display(canonical_event_counts)


Canonical strategic-state distribution:


,week_sector_rows
strategic_state,
BEARISH,2054
NORMAL_POSITIVE_TREND,1689
EARLY_REVERSAL,366
CONFIRMED_TREND,16


Canonical tactical-state distribution:


,week_sector_rows
tactical_state,
NONE,2942
PROFIT_TAKING,878
PULLBACK_ACCUMULATION,305


Canonical active-weight actions:


,week_sector_rows
active_weight_action,
SET_MAX_UNDERWEIGHT,2054
REDUCE_OVERWEIGHT,878
HOLD_POSITIVE_TILT,506
REBUILD_TOWARD_NEUTRAL,366
ADD_OVERWEIGHT,305
SET_NEUTRAL_ON_CONFIRMATION,16


Canonical event counts:


,event_count
start_early_reversal,17
end_early_reversal,17
start_pullback,69
end_pullback,77
start_profit_taking,85
end_profit_taking,81
position_entry_event,17
position_exit_event,8


In [11]:
# ============================================================
# BLOCK 4.11 — PARITY / ARCHITECTURE SAFETY CHECKS
# ============================================================

# Position entry and exit events must alternate logically by ticker.
for ticker, g in state_long.groupby("ticker"):
    g = g.sort_values("date")
    event_rows = g[
        g["position_entry_event"] | g["position_exit_event"]
    ]

    last = None
    for _, r in event_rows.iterrows():
        this = "ENTRY" if r["position_entry_event"] else "EXIT"

        if last == this:
            raise AssertionError(
                f"Non-alternating strategic events for {ticker}: {this}"
            )
        last = this

# Bearish strategic state must always map to maximum underweight.
bearish_rows = canonical_weights[
    canonical_weights["strategic_state"] == "BEARISH"
]
assert np.allclose(
    bearish_rows["active_multiplier"],
    ACTIVE_MIN,
)

# Effective profit taking must never create an underweight.
profit_rows = canonical_weights[
    canonical_weights["tactical_state"] == "PROFIT_TAKING"
]
assert (profit_rows["position_held"]).all()
assert (profit_rows["active_multiplier"] >= 0.0).all()

# Raw Pine profit-taking is allowed to exist while strategic position is OFF;
# those rows must be portfolio-inactive.
raw_profit_while_out = canonical_weights[
    (canonical_weights["pine_zone_state"] == "PROFIT_TAKING")
    & (~canonical_weights["position_held"])
]
assert (raw_profit_while_out["tactical_state"] == "NONE").all()
assert np.allclose(
    raw_profit_while_out["active_multiplier"],
    ACTIVE_MIN,
)

# Effective pullback accumulation must never coexist with bearish strategy.
invalid_pullback = canonical_weights[
    (canonical_weights["tactical_state"] == "PULLBACK_ACCUMULATION")
    & (canonical_weights["strategic_state"] == "BEARISH")
]
assert invalid_pullback.empty

# Raw Pine pullback zones while position is OFF are preserved for parity,
# but must not become portfolio-effective tactical states.
raw_pullback_while_out = canonical_weights[
    (canonical_weights["pine_zone_state"] == "PULLBACK_ACCUMULATION")
    & (~canonical_weights["position_held"])
]
assert (raw_pullback_while_out["tactical_state"] == "NONE").all()

# Early reversal cannot exceed neutral.
early_rows = canonical_weights[
    canonical_weights["strategic_state"] == "EARLY_REVERSAL"
]
assert (early_rows["active_multiplier"] <= 0.0).all()

# Raw multipliers remain long-only.
assert (
    canonical_weights["raw_weight_multiplier"]
    >= (1.0 + ACTIVE_MIN)
).all()

print("All Block 4 parity and architecture safety checks passed.")


All Block 4 parity and architecture safety checks passed.


In [12]:
# ============================================================
# BLOCK 4.12 — SAVE BLOCK 4 DATASETS
# ============================================================

STATE_LONG_PATH = (
    DIRS["data_processed"]
    / "weekly_state_dependent_state_long.parquet"
)

WEIGHT_LONG_PATH = (
    DIRS["data_processed"]
    / "weekly_state_dependent_active_weights_long.parquet"
)

TARGET_WEIGHT_WIDE_PATH = (
    DIRS["data_processed"]
    / "weekly_state_dependent_target_weights.parquet"
)

EVENT_COUNTS_PATH = (
    DIRS["tables"]
    / "block_4_canonical_event_counts.csv"
)

state_signal_long.to_parquet(
    STATE_LONG_PATH,
    index=False,
)

canonical_weights.to_parquet(
    WEIGHT_LONG_PATH,
    index=False,
)

target_weight_wide = (
    canonical_weights
    .pivot(
        index="date",
        columns="ticker",
        values="normalized_target_weight",
    )
    .reindex(columns=SECTOR_TICKERS)
)

target_weight_wide.to_parquet(
    TARGET_WEIGHT_WIDE_PATH
)

canonical_event_counts.to_csv(EVENT_COUNTS_PATH)

print("Saved:")
for p in [
    STATE_LONG_PATH,
    WEIGHT_LONG_PATH,
    TARGET_WEIGHT_WIDE_PATH,
    EVENT_COUNTS_PATH,
]:
    print(" ", p)


Saved:
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/weekly_state_dependent_state_long.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/weekly_state_dependent_active_weights_long.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/data/processed/weekly_state_dependent_target_weights.parquet
  /content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/outputs/tables/block_4_canonical_event_counts.csv


In [13]:
# ============================================================
# BLOCK 4.13 — SAVE BLOCK 4 MANIFEST
# ============================================================

block4_manifest = {
    "project": block1["project"],
    "block": (
        "Block 4 - State Machine & "
        "State-Dependent Active-Weight Engine"
    ),
    "created_utc": datetime.now(timezone.utc).isoformat(),

    "source_contract": {
        "indicator_source": block3["indicator_source"],
        "pine_zone_priority": (
            "EARLY_REVERSAL > PULLBACK_ACCUMULATION > PROFIT_TAKING"
        ),
        "strategic_state_separated_from_tactical_state": True,
        "raw_pine_zone_preserved_separately": True,
        "tactical_overlay_requires_position_held": True,
    },

    "active_weight_parameters": {
        "active_min": ACTIVE_MIN,
        "active_max": ACTIVE_MAX,
        "active_step": ACTIVE_STEP,
        "raw_multiplier_min": 1.0 + ACTIVE_MIN,
        "raw_multiplier_max": 1.0 + ACTIVE_MAX,
    },

    "state_definitions": {
        "strategic_states": [
            "BEARISH",
            "EARLY_REVERSAL",
            "CONFIRMED_TREND",
            "NORMAL_POSITIVE_TREND",
        ],
        "tactical_states": [
            "NONE",
            "PULLBACK_ACCUMULATION",
            "PROFIT_TAKING",
        ],
        "pine_zone_states": [
            "NEUTRAL",
            "EARLY_REVERSAL",
            "PULLBACK_ACCUMULATION",
            "PROFIT_TAKING",
        ],
    },

    "active_weight_rules": {
        "bearish": "Set active multiplier to -0.50",
        "early_reversal": (
            "Increase by +0.10 per week, capped at 0.00"
        ),
        "confirmed_trend": "Set active multiplier to 0.00",
        "pullback_accumulation": (
            "Increase by +0.10 per week, capped at +0.50"
        ),
        "profit_taking": (
            "Decrease positive overweight by 0.10 per week, floored at 0.00"
        ),
        "normal_positive_trend": (
            "Hold existing neutral/positive active multiplier"
        ),
    },

    "weight_formula": (
        "raw = strategic_weight * (1 + active_multiplier); "
        "normalized = raw / cross_sectional_raw_sum"
    ),

    "strategic_weight_method": block2["strategic_weight_method"],
    "canonical_signal_start": block3["canonical_signal_start"],
    "canonical_signal_end": block3["canonical_signal_end"],

    "lookahead_controls": [
        (
            "State machine processes weekly signals sequentially "
            "in chronological order for each sector."
        ),
        (
            "Active multiplier on a signal week is attached to the "
            "Block 3 next-session execution date."
        ),
        (
            "No future state, signal, weight, or return is used "
            "to determine current target weights."
        ),
    ],

    "saved_files": {
        "state_signal_long": str(STATE_LONG_PATH),
        "canonical_active_weights_long": str(WEIGHT_LONG_PATH),
        "canonical_target_weights_wide": str(TARGET_WEIGHT_WIDE_PATH),
        "canonical_event_counts": str(EVENT_COUNTS_PATH),
    },
}

BLOCK4_MANIFEST = (
    DIRS["manifests"] / "block_4_state_active_weights.json"
)

with open(BLOCK4_MANIFEST, "w", encoding="utf-8") as f:
    json.dump(block4_manifest, f, indent=2)

print("Saved Block 4 manifest:")
print(BLOCK4_MANIFEST)


Saved Block 4 manifest:
/content/drive/My Drive/Colab Notebooks/Sector Rotation Model/State-Dependent U.S. Equity Sector Rotation - A Systematic Framework for Trend-Based Active Sector Allocation/manifests/block_4_state_active_weights.json


In [14]:
# ============================================================
# BLOCK 4.14 — FINAL STATUS
# ============================================================

summary = pd.Series(
    {
        "Sector count": len(SECTOR_TICKERS),
        "Signal frequency": CONFIG["signal_frequency"],
        "Execution rule": CONFIG["rebalance_execution_rule"],
        "Strategic allocation": block2["strategic_weight_method"],

        "Active multiplier minimum": ACTIVE_MIN,
        "Active multiplier maximum": ACTIVE_MAX,
        "Weekly active step": ACTIVE_STEP,

        "Canonical signal start": canonical_start.date(),
        "Canonical signal end": canonical_end.date(),
        "Canonical rows": len(canonical_weights),
        "Canonical complete rows": int(canonical_complete.sum()),

        "Minimum realized active multiplier": (
            canonical_weights["active_multiplier"].min()
        ),
        "Maximum realized active multiplier": (
            canonical_weights["active_multiplier"].max()
        ),

        "Strategic entry events": int(
            canonical_weights["position_entry_event"].sum()
        ),
        "Strategic exit events": int(
            canonical_weights["position_exit_event"].sum()
        ),
        "Early-reversal starts": int(
            canonical_weights["start_early_reversal"].sum()
        ),
        "Pullback starts": int(
            canonical_weights["start_pullback"].sum()
        ),
        "Profit-taking starts": int(
            canonical_weights["start_profit_taking"].sum()
        ),

        "Target weights sum to 1": bool(
            np.allclose(target_sums.values, 1.0, atol=1e-10)
        ),
        "Long-only target weights": bool(
            (canonical_weights["normalized_target_weight"] >= 0).all()
        ),
    },
    name="Block 4 status",
).to_frame()

display(summary)

print("\nBLOCK 4 COMPLETE")
print(
    "Next: Block 5 — Benchmark Active Weights & Portfolio Engine"
)


,Block 4 status
Sector count,11
Signal frequency,W-FRI
Execution rule,NEXT_US_TRADING_SESSION_AFTER_SIGNAL
Strategic allocation,INVERSE_VOLATILITY_52W
Active multiplier minimum,-0.5
Active multiplier maximum,0.5
Weekly active step,0.1
Canonical signal start,2019-06-21
Canonical signal end,2026-08-21
Canonical rows,4125



BLOCK 4 COMPLETE
Next: Block 5 — Benchmark Active Weights & Portfolio Engine
